In [15]:
# import libraries
from pathlib import Path
from zipfile import ZipFile
from datetime import datetime, timedelta
import re
from collections import defaultdict
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd

In [ ]:
# File paths
DATA_DIR = Path("C:/Users/htunlong/OneDrive - UGent/Research/Postdoc/Fundings/VMM/Data/Data_Module_5/Module 3/CSOs")

EK_FILE = DATA_DIR / "EK files samengevoegd.xlsx"
OVE_FILE = DATA_DIR / "OVE files samengevoegd.xlsx"
LOZINGEN_FILE = DATA_DIR / "Lozingspunten (1).xlsx"

OUTPUT_FILE = DATA_DIR / "M5_step03_CSO_source_catalogue_Eksel_Overpelt.xlsx"


In [21]:
# Read and validate input files

cso_locations_raw = pd.read_excel(
    LOZINGEN_FILE,
    sheet_name="CSO",
)

print("Raw CSO location table shape:", cso_locations_raw.shape)
print(cso_locations_raw.columns.tolist())

cso_locations_raw.head()


Raw CSO location table shape: (82, 9)
['xlsx-bestandsnaam', 'Naam', 'Zuiv.gebied', 'x(m)', 'y(m)', 'Ontvangende waterloop', 'Deelbekken', 'Koppeling in ICM ? ', 'Opmerkingen']


,xlsx-bestandsnaam,Naam,Zuiv.gebied,x(m),y(m),Ontvangende waterloop,Deelbekken,Koppeling in ICM ?,Opmerkingen
0,PE_21360_3_2,OS BBB Bomerstraat,Peer,225212.0,202747.0,Dommel,Dommel,N,NaN
1,PE_F428001_1,OS Dijkerstraat,Peer,224509.2,203228.5,Kleine Beek,Dommel,J,NaN
2,PE_K306802_1,OS Meeuwerbaan,Peer,225848.2,197200.8,Dommel,Dommel,N,NaN
3,PE_K427503_1,OS Steenweg Wijchmaal,Peer,224996.6,202967.8,Dommel,Dommel,J,NaN
4,PE_K427602_1,OS Croxdijk,Peer,226026.9,200337.2,Dommel,Dommel,N,NaN


In [23]:
# Keep only Eksel and Overpelt CSO locations

cso_locations = cso_locations_raw.rename(columns={
    "xlsx-bestandsnaam": "source_id_raw",
    "Naam": "source_name",
    "Zuiv.gebied": "site",
    "x(m)": "x_m",
    "y(m)": "y_m",
    "Ontvangende waterloop": "receiving_watercourse",
    "Deelbekken": "subbasin",
    "Koppeling in ICM ? ": "coupled_in_icm",
    "Opmerkingen": "remarks",
})

cso_locations = cso_locations[
    cso_locations["site"].isin(["Eksel", "Overpelt"])
].copy()

print("Filtered CSO locations:", cso_locations.shape)
print(cso_locations["site"].value_counts())

Filtered CSO locations: (37, 9)
site
Overpelt    26
Eksel       11
Name: count, dtype: int64


In [24]:
# Split combined CSO IDs into separate rows

def split_source_ids(value):
    """
    Split cells that contain multiple source IDs.

    Example:
    'EK_6019_2 / EK_6020_2'
    becomes:
    ['EK_6019_2', 'EK_6020_2']
    """
    if pd.isna(value):
        return []

    parts = re.split(r"/|\n", str(value))
    clean_parts = []

    for part in parts:
        part = part.strip()

        if part:
            clean_parts.append(part)

    return clean_parts


location_rows = []

for _, row in cso_locations.iterrows():
    source_ids = split_source_ids(row["source_id_raw"])

    for source_id in source_ids:
        location_rows.append({
            "site": row["site"],
            "location_source_id": source_id,
            "source_name": row["source_name"],
            "x_m": row["x_m"],
            "y_m": row["y_m"],
            "receiving_watercourse": row["receiving_watercourse"],
            "subbasin": row["subbasin"],
            "coupled_in_icm": row["coupled_in_icm"],
            "remarks": row["remarks"],
        })

cso_locations_clean = pd.DataFrame(location_rows)

print("Clean CSO location rows:", len(cso_locations_clean))
print(cso_locations_clean["site"].value_counts())

Clean CSO location rows: 39
site
Overpelt    27
Eksel       12
Name: count, dtype: int64


In [25]:
# Create a simple matching lookup key based on the source ID

def make_match_key(value):
    """
    Create a stable ID for matching CSO time-series IDs
    to CSO location IDs.

    This handles small formatting differences:
    - underscores
    - commas
    - O1ov versus 01ov
    - one technical suffix at the end
    """
    if pd.isna(value):
        return ""

    text = str(value).strip().upper()

    # Known text inconsistencies.
    text = text.replace("0VE", "OVE")
    text = text.replace("O1OV", "01OV")
    text = text.replace(",", "")

    # Remove final technical suffix only where appropriate.
    # Examples:
    # EK_1128_1              -> EK_1128
    # EK1168_2               -> EK1168
    # OVE_114_ouduit_t_10295 -> OVE_114_ouduit_t
    has_two_or_more_underscores = text.count("_") >= 2
    has_prefix_digits_then_suffix = re.match(
        r"^[A-Z]+[0-9][0-9A-Z]*_\d+$",
        text,
    ) is not None

    if has_two_or_more_underscores or has_prefix_digits_then_suffix:
        text = re.sub(r"_\d+$", "", text)

    # Known small ID-format difference in the Overpelt file.
    text = text.replace("K000010", "K00010")

    # Final key: keep only letters and numbers.
    text = re.sub(r"[^A-Z0-9]", "", text)

    return text


cso_locations_clean["match_key"] = (
    cso_locations_clean["location_source_id"].apply(make_match_key)
)

cso_locations_clean[
    ["site", "location_source_id", "match_key", "source_name", "receiving_watercourse"]
].head(15)

,site,location_source_id,match_key,source_name,receiving_watercourse
0,Eksel,EK_1128_1,EK1128,OL Zavelstraat,Dorperloop
1,Eksel,EK_1168_2,EK1168,OS Winnerstraat,Gortenloop
2,Eksel,EK_1830_1,EK1830,OS Lijsterstraat,Bollisenbeek
3,Eksel,EK_5071_1,EK5071,OS Kenensdijk,Bollisenbeek
4,Eksel,EK_6019_2,EK6019,OS Tichelovenstraat,Bollisenbeek
5,Eksel,EK_6020_2,EK6020,OS Tichelovenstraat,Bollisenbeek
6,Eksel,EK_7023_1,EK7023,OS Mortelkensstraat,Prinsenloop
7,Eksel,EK_7050_1,EK7050,OS Overweglaan,Bollisenbeek
8,Eksel,EK_7111_1,EK7111,OS BBB Dommellaan,Dommel
9,Eksel,EK_10849_1,EK10849,OS Heufkensstraat,Dorperloop


In [27]:
# Helper to read only source ID, time, and Q from a CSO workbook

EXCEL_ZERO_DATE = datetime(1899, 12, 30)


def read_shared_strings(xlsx_file):
    """
    Read Excel shared strings.

    Excel stores text values separately inside the .xlsx file.
    This helper converts those text indexes back to text.
    """
    with ZipFile(xlsx_file) as zip_file:
        if "xl/sharedStrings.xml" not in zip_file.namelist():
            return []

        xml_text = zip_file.read("xl/sharedStrings.xml")

    root = ET.fromstring(xml_text)

    strings = []

    for item in root:
        text_parts = []

        for node in item.iter():
            if node.tag.endswith("t") and node.text:
                text_parts.append(node.text)

        strings.append("".join(text_parts))

    return strings


def find_sheet_xml_path(xlsx_file, sheet_name):
    """
    Find the internal XML file for a worksheet name.
    """
    namespace = {
        "main": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
        "rel": "http://schemas.openxmlformats.org/package/2006/relationships",
    }

    with ZipFile(xlsx_file) as zip_file:
        workbook_xml = ET.fromstring(zip_file.read("xl/workbook.xml"))
        rels_xml = ET.fromstring(zip_file.read("xl/_rels/workbook.xml.rels"))

    rel_targets = {}

    for rel in rels_xml:
        rel_targets[rel.attrib["Id"]] = rel.attrib["Target"]

    sheets = workbook_xml.find("main:sheets", namespace)

    for sheet in sheets:
        if sheet.attrib["name"] == sheet_name:
            rel_id = sheet.attrib[
                "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"
            ]
            return "xl/" + rel_targets[rel_id]

    raise ValueError(f"Sheet not found: {sheet_name}")


def get_cell_value(cell_attributes, raw_value, shared_strings):
    """
    Convert an Excel cell value to normal text/number.
    """
    if raw_value is None:
        return None

    value = raw_value.decode("utf-8")

    if b't="s"' in cell_attributes:
        return shared_strings[int(value)]

    return value


def excel_time_to_timestamp(value):
    """
    Convert Excel time values to pandas Timestamp.
    """
    try:
        return pd.Timestamp(EXCEL_ZERO_DATE + timedelta(days=float(value)))
    except Exception:
        return pd.to_datetime(value, errors="coerce")


def read_cso_timeseries_summary(xlsx_file, sheet_name, site):
    """
    Read one CSO workbook and summarise each source ID.

    This reads only source ID, time, and Q.
    """
    shared_strings = read_shared_strings(xlsx_file)
    sheet_xml_path = find_sheet_xml_path(xlsx_file, sheet_name)

    with ZipFile(xlsx_file) as zip_file:
        sheet_xml = zip_file.read(sheet_xml_path)

    row_pattern = re.compile(rb'<row[^>]*r="(\d+)"[^>]*>(.*?)</row>')
    cell_pattern = re.compile(rb'<c r="([ABC]+)\d+"([^>]*)>(?:<v>(.*?)</v>)?</c>')

    summaries = {}

    for row_match in row_pattern.finditer(sheet_xml):
        row_number = int(row_match.group(1))

        # Row 1 is headers, row 2 is units.
        if row_number < 3:
            continue

        row_xml = row_match.group(2)
        cells = {}

        for cell_match in cell_pattern.finditer(row_xml):
            column_letter = cell_match.group(1).decode("ascii")
            cell_attributes = cell_match.group(2)
            raw_value = cell_match.group(3)

            cells[column_letter] = get_cell_value(
                cell_attributes,
                raw_value,
                shared_strings,
            )

        source_id = cells.get("A")
        time_value = cells.get("B")
        q_value = cells.get("C")

        if source_id is None or time_value is None:
            continue

        time = excel_time_to_timestamp(time_value)

        if pd.isna(time):
            continue

        if time < pd.Timestamp("2017-01-01"):
            continue

        if time >= pd.Timestamp("2018-01-01"):
            continue

        source_id = str(source_id).strip().replace(",", "")
        q = 0.0 if q_value is None else float(q_value)

        if source_id not in summaries:
            summaries[source_id] = {
                "site": site,
                "timeseries_source_id": source_id,
                "source_file": Path(xlsx_file).name,
                "source_sheet": sheet_name,
                "first_time_2017": time,
                "last_time_2017": time,
                "rows_2017": 0,
                "active_rows_q_gt_0": 0,
                "total_volume_m3_2017": 0.0,
                "max_q_m3s_2017": q,
            }

        summary = summaries[source_id]

        summary["first_time_2017"] = min(summary["first_time_2017"], time)
        summary["last_time_2017"] = max(summary["last_time_2017"], time)
        summary["rows_2017"] += 1
        summary["total_volume_m3_2017"] += q * 900
        summary["max_q_m3s_2017"] = max(summary["max_q_m3s_2017"], q)

        if q > 0:
            summary["active_rows_q_gt_0"] += 1

    result = pd.DataFrame(summaries.values())
    result["match_key"] = result["timeseries_source_id"].apply(make_match_key)

    return result

In [28]:
# Read Eksel and Overpelt CSO time-series summaries

eksel_timeseries = read_cso_timeseries_summary(
    EK_FILE,
    sheet_name="alle data",
    site="Eksel",
)

overpelt_timeseries = read_cso_timeseries_summary(
    OVE_FILE,
    sheet_name="Blad1",
    site="Overpelt",
)

cso_timeseries_summary = pd.concat(
    [eksel_timeseries, overpelt_timeseries],
    ignore_index=True,
)

print("CSO time-series IDs:", len(cso_timeseries_summary))
print(cso_timeseries_summary["site"].value_counts())

CSO time-series IDs: 1280
site
Overpelt    1268
Eksel         12
Name: count, dtype: int64
